
## Eulerian Path in Directed Graphs

**Course:** Genome Sequencing (Bioinformatics II)  
**Institution:** University of California San Diego — Coursera  
**Language:** Python 3  
**Topic:** Graph Algorithms and Genome Assembly

###  Overview

This notebook explores the construction of Eulerian paths in directed graphs, a fundamental concept in graph-based genome assembly.

An Eulerian path is a traversal that visits every edge of a graph exactly once. In bioinformatics, this concept is particularly relevant to genome reconstruction using De Bruijn graphs, where edges represent k-mers and nodes represent their overlapping prefixes and suffixes.

###  Objectives

- Parse directed graphs from adjacency lists.
- Calculate the in-degree and out-degree of graph nodes.
- Identify potential starting and ending nodes.
- Construct Eulerian paths and cycles using a stack-based algorithm.
- Validate graph connectivity and Eulerian degree conditions.
- Verify that the selected starting node is valid.
- Ensure that every edge is traversed exactly once, including repeated edges.
- Preserve the original graph during traversal.

### Implementation

The notebook implements a stack-based approach based on **Hierholzer's algorithm** for constructing Eulerian paths.

The implementation includes input validation, graph traversal, output formatting, and tests covering valid and invalid graph structures.

###  Learning Context

This implementation was originally developed as part of the *Genome Sequencing* course in the Bioinformatics Specialization offered by UC San Diego on Coursera.

The code was subsequently reviewed, documented, and extended with additional validation checks and tests to improve its robustness and reproducibility.

###  Requirements

- Python 3
- Python standard library (`collections`)

No external Python packages are required for the algorithm.

###  Validation

The notebook includes 11 tests covering basic traversal, Eulerian cycles, terminal nodes, graph connectivity, degree conditions, starting node validation, repeated edges, and path completeness.

These tests verify selected cases and do not constitute exhaustive formal verification.

In [5]:

from collections import defaultdict
from collections import Counter


def parse_adjacency_list(adj_list):
    """
    Convert an adjacency list from strings into a dictionary.

    Parameters
    ----------
    adj_list : list[str]
        Lines describing directed edges, such as "0: 1 2".

    Returns
    -------
    defaultdict[int, list[int]]
        Graph represented as an adjacency list.
    """
    graph = defaultdict(list)

    for line in adj_list:
        node, neighbors = line.split(": ")
        graph[int(node)] = list(map(int, neighbors.strip().split()))

    return graph


def calculate_degrees(graph):
    """
    Calculate the out-degree and in-degree of each node.
    """
    out_degrees = {node: len(edges) for node, edges in graph.items()}
    in_degrees = defaultdict(int)

    for edges in graph.values():
        for node in edges:
            in_degrees[node] += 1

    # Include all source nodes, even if their in-degree is zero
    for node in graph:
        if node not in in_degrees:
            in_degrees[node] = 0

    return out_degrees, dict(in_degrees)


def find_unbalanced_nodes(out_degrees, in_degrees):
    """
    Identify potential starting and ending nodes
    based on the difference between their degrees.
    """
    start_node = None
    end_node = None

    for node in set(out_degrees).union(in_degrees):
        out_diff = (
            out_degrees.get(node, 0)
            - in_degrees.get(node, 0)
        )

        if out_diff > 0:
            start_node = node  # Starting node
        elif out_diff < 0:
            end_node = node  # Ending node

    return start_node, end_node


def validate_eulerian_degrees(graph):
    """
    Check whether a directed graph satisfies the
    degree conditions for an Eulerian path or cycle.

    Raises
    ------
    ValueError
        If the graph violates Eulerian degree conditions.
    """
    out_degrees, in_degrees = calculate_degrees(graph)

    all_nodes = set(out_degrees) | set(in_degrees)

    start_candidates = 0
    end_candidates = 0

    for node in all_nodes:
        difference = (
            out_degrees.get(node, 0)
            - in_degrees.get(node, 0)
        )

        if difference == 1:
            start_candidates += 1
        elif difference == -1:
            end_candidates += 1
        elif difference != 0:
            raise ValueError(
                "Graph does not satisfy Eulerian degree conditions."
            )

    valid_cycle = (
        start_candidates == 0
        and end_candidates == 0
    )

    valid_path = (
        start_candidates == 1
        and end_candidates == 1
    )

    if not (valid_cycle or valid_path):
        raise ValueError(
            "Graph does not satisfy Eulerian degree conditions."
        )


def validate_graph_connectivity(graph):
    """
    Check whether all nodes with at least one edge belong
    to the same weakly connected component.

    Edge directions are ignored for connectivity checking.

    Raises
    ------
    ValueError
        If the graph contains disconnected components
        with edges.
    """

    # Build an undirected representation of the graph
    undirected_graph = defaultdict(set)

    for node, neighbors in graph.items():
        for neighbor in neighbors:
            undirected_graph[node].add(neighbor)
            undirected_graph[neighbor].add(node)

    # An empty graph has no disconnected edges
    if not undirected_graph:
        return

    # Start from any node with at least one incident edge
    start_node = next(iter(undirected_graph))

    visited = set()
    stack = [start_node]

    # Explore all reachable nodes
    while stack:
        current_node = stack.pop()

        if current_node in visited:
            continue

        visited.add(current_node)

        for neighbor in undirected_graph[current_node]:
            if neighbor not in visited:
                stack.append(neighbor)

    # Every node with incident edges must be reachable
    if visited != set(undirected_graph):
        raise ValueError(
            "Graph contains disconnected components with edges."
        )


def validate_start_node(graph, start_node):
    """
    Validate the starting node for an Eulerian traversal.

    For an open Eulerian path, the starting node must
    have out-degree exactly one greater than in-degree.

    For an Eulerian cycle, any node with outgoing edges
    can be used as the starting node.

    Raises
    ------
    ValueError
        If the selected starting node is invalid.
    """
    out_degrees, in_degrees = calculate_degrees(graph)

    # A traversal requires at least one edge
    total_edges = sum(out_degrees.values())

    if total_edges == 0:
        raise ValueError("Graph contains no edges.")

    # Find the required starting node for an open path
    required_start = None

    for node in set(out_degrees) | set(in_degrees):
        difference = (
            out_degrees.get(node, 0)
            - in_degrees.get(node, 0)
        )

        if difference == 1:
            required_start = node
            break

    if required_start is not None:
        if start_node != required_start:
            raise ValueError(
                f"Invalid starting node. "
                f"Expected node {required_start}."
            )

    else:
        # For a cycle, the starting node needs outgoing edges
        if out_degrees.get(start_node, 0) == 0:
            raise ValueError(
                "Starting node must have outgoing edges."
            )


def validate_eulerian_path(graph, path):
    """
    Verify that a path traverses every graph edge
    exactly once, preserving edge multiplicities.

    Raises
    ------
    ValueError
        If the path does not match the original edges.
    """

    # Count the original directed edges
    original_edges = Counter(
        (node, neighbor)
        for node, neighbors in graph.items()
        for neighbor in neighbors
    )

    # Count the edges traversed by the path
    traversed_edges = Counter(
        zip(path, path[1:])
    )

    if original_edges != traversed_edges:
        raise ValueError(
            "Constructed path does not traverse "
            "every edge exactly once."
        )


def EulerianPath(graph, start_node):
    """
    Construct an Eulerian path in a directed graph.

    Parameters
    ----------
    graph : dict[int, list[int]]
        Adjacency list representing a directed graph.
    start_node : int
        Starting node for the traversal.

    Returns
    -------
    list[int]
        Sequence of nodes representing the path.

    Notes
    -----
    The algorithm uses a stack-based traversal and
    preserves the original graph by working on a copy.

    Eulerian degree conditions, weak connectivity,
    and the selected starting node are validated
    before traversal.
    """

    # Validate Eulerian degree conditions
    validate_eulerian_degrees(graph)

    # Validate graph connectivity
    validate_graph_connectivity(graph)

    # Vakudate the selected starting node
    validate_start_node(graph, start_node)

    # Copy the graph to preserve its original edges
    graph_copy = {
        node: edges[:]
        for node, edges in graph.items()
    }

    # Include destination-only nodes with no outgoing edges
    for edges in graph.values():
        for node in edges:
            graph_copy.setdefault(node, [])

    current_path = [start_node]
    path = []

    # Construct the path
    while current_path:
        current_node = current_path[-1]

        if graph_copy[current_node]:
            next_node = graph_copy[current_node].pop()
            current_path.append(next_node)
        else:
            path.append(current_path.pop())

    
    # Reverse the constructed path
    final_path = path[::-1]

    # Verify that every original edge was traversed
    validate_eulerian_path(graph, final_path)

    return final_path


def format_output(path):
    """
    Format the Eulerian path as a space-separated string.
    """
    return " ".join(map(str, path))


def main():
    """
    Read a graph from a dataset and construct
    its Eulerian path.
    """

    # Read the input file
    input_file = (
        "Dataset/dataset_30187_6.txt"
    )

    with open(input_file, "r") as f:
        adj_list = f.readlines()

    # Parse the graph
    graph = parse_adjacency_list(adj_list)

    # Calculate node degrees
    out_degrees, in_degrees = calculate_degrees(graph)

    # Identify potential starting and ending nodes
    start_node, end_node = find_unbalanced_nodes(
        out_degrees,
        in_degrees
    )

    # Select a starting node for an Eulerian cycle
    if start_node is None:
        if not graph:
            print("The graph is empty.")
            return

        start_node = next(iter(graph))

    # Construct the Eulerian path
    path = EulerianPath(graph, start_node)

    # Format and print the result
    formatted_path = format_output(path)
    print(formatted_path)


# Run the main program
if __name__ == "__main__":
    main()

4 5 0 1 2 8 9 7 2 3 0


## Validation and Testing

This section contains a series of tests designed to evaluate the correctness and robustness of the Eulerian path implementation.

The tests cover:

1. **Basic path construction:** Verifies the reconstruction of a simple Eulerian path.
2. **Terminal node handling:** Ensures that nodes without outgoing edges are processed correctly.
3. **Eulerian cycle detection:** Checks the construction of a valid Eulerian cycle.
4. **Input preservation:** Confirms that the original graph remains unchanged during traversal.
5. **Output formatting:** Verifies the representation of the resulting path.
6. **Degree validation:** Rejects graphs that do not satisfy Eulerian degree conditions.
7. **Connectivity validation:** Detects disconnected components containing edges.
8. **Invalid starting node:** Rejects an incorrect starting node.
9. **Valid starting node:** Confirms that a valid starting node is accepted.
10. **Repeated edges:** Ensures that edge multiplicities are preserved during traversal.
11. **Incomplete path detection:** Verifies that paths missing original edges are rejected.

These tests use Python's built-in `assert` statements and exception handling to compare actual results against expected behavior.

**Note:** Successful execution of these tests provides evidence of correctness for the selected cases but does not constitute exhaustive verification of the algorithm.


In [6]:
# Test 1: Simple Eulerian path
graph = {
    0: [1],
    1: [2]
}

result = EulerianPath(graph, 0)

assert result == [0, 1, 2]
print("Test 1 passed: Simple Eulerian path")


# Test 2: Destination-only node
graph = {
    0: [1],
    1: [2],
    2: [3]
}

result = EulerianPath(graph, 0)

assert result == [0, 1, 2, 3]
print("Test 2 passed: Destination-only node")


# Test 3: Eulerian cycle
graph = {
    0: [1],
    1: [2],
    2: [0]
}

result = EulerianPath(graph, 0)

assert result == [0, 1, 2, 0]
print("Test 3 passed: Eulerian cycle")


# Test 4: Original graph must remain unchanged
graph = {
    0: [1],
    1: [2]
}

original_graph = {
    node: edges[:]
    for node, edges in graph.items()
}

EulerianPath(graph, 0)

assert graph == original_graph
print("Test 4 passed: Original graph preserved")


# Test 5: Output formatting
path = [0, 1, 2, 3]

assert format_output(path) == "0 1 2 3"
print("Test 5 passed: Output formatting")


# Test 6: Reject a graph with invalid Eulerian degrees
graph = {
    0: [1, 2],
    1: [],
    2: []
}

try:
    EulerianPath(graph, 0)
except ValueError as error:
    print("Test 6 passed:", error)
else:
    raise AssertionError(
        "Expected ValueError for invalid Eulerian degrees"
    )


# Test 7: Reject disconnected components with edges
graph = {
    0: [1],
    1: [0],
    2: [3],
    3: [2]
}

try:
    EulerianPath(graph, 0)
except ValueError as error:
    assert "disconnected" in str(error).lower()
    print("Test 7 passed: Disconnected graph rejected")
else:
    raise AssertionError(
        "Expected ValueError for disconnected components"
    )


# Test 8: Reject an invalid starting node
graph = {
    0: [1],
    1: [2],
    2: []
}

try:
    EulerianPath(graph, 1)
except ValueError as error:
    assert "starting node" in str(error).lower()
    print("Test 8 passed: Invalid starting node rejected")
else:
    raise AssertionError(
        "Expected ValueError for invalid starting node"
    )


# Test 9: Accept the correct starting node
result = EulerianPath(graph, 0)

assert result == [0, 1, 2]
print("Test 9 passed: Valid starting node accepted")


# Test 10: Validate repeated edges
graph = {
    0: [1, 1],
    1: [0]
}

result = EulerianPath(graph, 0)

assert result == [0, 1, 0, 1]
print("Test 10 passed: Repeated edges preserved")


# Test 11: Reject an incomplete path
graph = {
    0: [1],
    1: [2],
    2: []
}

invalid_path = [0, 1]

try:
    validate_eulerian_path(graph, invalid_path)
except ValueError:
    print("Test 11 passed: Incomplete path rejected")
else:
    raise AssertionError(
        "Expected ValueError for incomplete path"
    )

Test 1 passed: Simple Eulerian path
Test 2 passed: Destination-only node
Test 3 passed: Eulerian cycle
Test 4 passed: Original graph preserved
Test 5 passed: Output formatting
Test 6 passed: Graph does not satisfy Eulerian degree conditions.
Test 7 passed: Disconnected graph rejected
Test 8 passed: Invalid starting node rejected
Test 9 passed: Valid starting node accepted
Test 10 passed: Repeated edges preserved
Test 11 passed: Incomplete path rejected
